# RealSaS — Geppetto Reference-Strength FIT1 V1

Fail-closed one-run notebook. This notebook does **not** imply promotion or generalization. It runs the frozen FIT1 capability gate on the pinned Mage witness.


In [ ]:
!pip -q install google-api-python-client google-auth-httplib2 google-auth-oauthlib
from google.colab import auth, drive, userdata
auth.authenticate_user()
drive.mount('/content/drive')
print('COLAB_AUTH_READY')


In [ ]:
import os, pathlib, subprocess, shutil, json, hashlib
TOKEN = userdata.get('GITHUB_TOKEN')
if not TOKEN:
    raise RuntimeError('Set a Colab secret named GITHUB_TOKEN with read access to merynz/RealSaS-OPT')
REPO = pathlib.Path('/content/RealSaS-OPT')
if REPO.exists(): shutil.rmtree(REPO)
url = f'https://{TOKEN}@github.com/merynz/RealSaS-OPT.git'
subprocess.run(['git','clone','--depth','1','--branch','exp/geppetto-reference-strength-fullstack-v1-20260907',url,str(REPO)], check=True)
subprocess.run(['git','-C',str(REPO),'remote','set-url','origin','https://github.com/merynz/RealSaS-OPT.git'], check=True)
del TOKEN, url
os.chdir(REPO)
print('REPO_HEAD=', subprocess.check_output(['git','rev-parse','HEAD'], text=True).strip())


In [ ]:
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
import google.auth, io
creds, _ = google.auth.default()
svc = build('drive','v3',credentials=creds)
INP = pathlib.Path('/content/realsas_fit1_inputs'); INP.mkdir(exist_ok=True)
def dl(file_id, name):
    path = INP/name
    req = svc.files().get_media(fileId=file_id)
    with io.FileIO(path,'wb') as fh:
        downloader = MediaIoBaseDownload(fh, req)
        done=False
        while not done:
            _, done = downloader.next_chunk()
    return path
ZERO = dl('1ov5QL3H4nVNDAQmVI4j0qcr5iOmC7-cT','ZERO_SURFACE_PRODUCT_CLIPPED.npz')
CORPUS = dl('1KjSA9ccjf-ZuXzAp6WeJYfwjxCdsQbj_','normalized.npz')
CAM_IDS = ['14QkLJkZlE7RIPkeWlwKsnZhHmSj-uH9F','1bNhqKxvV5tBHvXqepD5YGD4SOEaaBHUP','1N6bAQy6k647JY8pbBS5hFNfsdTaUwWBM','1wMkF-NCNqXI47KVCzOb2hGxAu4SoyXNp','1zR4EWLZFMl5eeXtyC1scIbYE2PfHLuv-','1ENCFAj5IwG9u0YADUI_deNEzdhSNDEw2','1sJoMeDS2_LVkmrPDxRY_cF9bS_4xvi7n','1Gv--1f3WpZ-nw5u9XIMPeNPSgnL8K0X1']
CAMS = [dl(fid,f'V{i}.camera.json') for i,fid in enumerate(CAM_IDS)]
def sha(p): return hashlib.sha256(pathlib.Path(p).read_bytes()).hexdigest()
assert sha(ZERO) == '987f7d18ce202454c4ea5101225bfaed54aeb4638cba1077e70efc15f2038e9b'
assert sha(CORPUS) == '528bef491eceb358ebc8ecb2a46af1d37b4322a7ef500281403a8207fe7c648f'
print('PINNED_INPUTS_READY')


In [ ]:
subprocess.run(['python','-m','py_compile',
 'experiments/geppetto_reference_strength_fullstack_v1/rigging_surface_tensorization_v1.py',
 'experiments/geppetto_reference_strength_fullstack_v1/geppetto_reference_strength_candidate_v1.py',
 'experiments/geppetto_reference_strength_fullstack_v1/geppetto_reference_strength_no_learned_slot_v1.py',
 'experiments/geppetto_reference_strength_fullstack_v1/mechanical_core_target_v1.py',
 'experiments/geppetto_reference_strength_fullstack_v1/geppetto_reference_strength_loss_v1.py',
 'experiments/geppetto_reference_strength_fullstack_v1/run_geppetto_reference_strength_fit1_v1.py'], check=True)
subprocess.run(['python','-m','pytest','-q',
 'tests/experiments/test_geppetto_reference_strength_mechanical_core_target_v1.py',
 'tests/experiments/test_geppetto_reference_strength_no_learned_slot_v1.py'], check=True)
print('SOURCE_PREFLIGHT_PASS')


In [ ]:
OUT = pathlib.Path('/content/drive/MyDrive/RealSaS_MAGE_GEPPETTO_REFERENCE_STRENGTH_FIT1_V1')
OUT.mkdir(parents=True, exist_ok=True)
cmd = ['python','experiments/geppetto_reference_strength_fullstack_v1/run_geppetto_reference_strength_fit1_v1.py',
 '--zero-surface',str(ZERO),'--normalized-corpus',str(CORPUS),'--cameras',*[str(x) for x in CAMS],
 '--output-dir',str(OUT)]
print('RUNNING=', ' '.join(cmd))
subprocess.run(cmd, check=True)
print('FIT1_RUN_COMPLETE', OUT)


In [ ]:
result = json.loads((OUT/'GEPPETTO_REFERENCE_STRENGTH_FIT1_RESULT.json').read_text())
print(json.dumps({k:result.get(k) for k in ['status','repo_head','architecture_id','closure_step','terminal_streak','surface_node_count','surface_edge_count','target_content_sha256','checkpoint_sha256']}, indent=2))
from IPython.display import display, Image
png = OUT/'GEPPETTO_REFERENCE_STRENGTH_FIT1_SKELETON.png'
if png.exists(): display(Image(filename=str(png)))
